In [6]:
import pandas as pd 
import numpy as np 


from sklearn.model_selection import StratifiedKFold, train_test_split, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import matplotlib.pyplot as plt
import seaborn as sns


sns.set_style("whitegrid")

%matplotlib inline

In [7]:
df = pd.read_csv("../data/interim/appendicitis_clean.csv")
print(df.shape)
display(df.head(20))

(1500, 20)


,Age,Gender,BMI,Is_Pregnant,Duration_of_Symptoms_Hours,Pain_Migration,Abdominal_Pain_Location,Nausea_Vomiting,Loss_of_Appetite,Fever_Temp_C,Rebound_Tenderness,McBurney_Sign,Rovsing_Sign,Psoas_Sign,WBC_Count_k_uL,Neutrophil_Percentage,CRP_Level_mg_L,Ultrasound_Findings,Final_Diagnosis,target_bin
0,32,Female,23.4,0,17,1,RLQ,1,0,37.7,1,1,1,1,11.5,77.8,63.2,Non-visualized,Appendicitis,1
1,5,Male,19.4,0,31,0,RLQ,0,0,38.2,1,1,1,0,12.6,74.7,89.1,Target Sign,Appendicitis,1
2,28,Female,27.5,0,49,1,RLQ,1,1,37.7,1,1,0,1,16.2,77.0,92.9,Periappendiceal Fluid,Appendicitis,1
3,20,Male,28.7,0,32,0,RLQ,1,1,37.5,1,1,1,1,19.1,88.3,12.7,Periappendiceal Fluid,Appendicitis,1
4,12,Female,22.0,0,47,0,RLQ,0,0,37.3,0,0,1,0,12.2,65.0,8.8,Normal,Other (Ectopic Pregnancy (if pregnant)),0
5,21,Female,20.4,0,43,1,RLQ,1,1,37.5,1,0,1,0,11.3,81.5,34.8,Appendicolith Seen,Appendicitis,1
6,57,Male,28.9,0,19,1,Generalized,0,1,38.7,1,1,0,0,11.2,78.8,28.0,Enlarged (>6mm),Appendicitis,1
7,41,Female,28.7,0,28,1,Generalized,1,1,38.5,1,0,1,1,16.9,87.5,70.6,Periappendiceal Fluid,Appendicitis,1
8,21,Female,26.3,0,8,1,Generalized,1,1,37.9,1,1,1,0,14.4,82.9,199.8,Target Sign,Appendicitis,1
9,5,Female,20.6,0,29,0,RLQ,1,1,38.0,0,1,0,1,20.9,79.1,42.0,Enlarged (>6mm),Appendicitis,1


In [9]:
target_col = "Final_Diagnosis"
df["target_bin"] = (df[target_col] == "Appendicitis").astype(int)

In [ ]:
df_feat = df.copy()

df_feat["Age_Group"] = pd.cut(
    df_feat["Age"],
    bins=[0, 12, 18, 35, 60, 120],
    labels=["Child", "Teen", "Young Adult", "Adult", "Senior"],
    include_lowest =True
)


df_feat["BMI_Group"] = pd.cut(
    df_feat["BMI"],
    bins=[0, 18.5, 25, 30, 100],
    labels=["Underweight", "Normal", "Overweight", "Obese"],
    include_lowest=True
)


df_feat["Symptoms_Positive_Count"] = (
    df_feat[["Pain_Migration", "Nausea_Vomiting", "Loss_of_Appetite", "Rebound_Tenderness", "McBurney_Sign", "Rovsing_Sign", "ign"]].sum(axis=1)
)

df_feat["Inflamation_Score"] = (
    df_feat["WBC_Count_k_uL"] + df_feat["Neutrophil_Percentage"] / 10  + df_feat["CRP_Level_mg_L"]
)

df_feat["Duration_Bucket"] = pd.cut(
    df_feat["Duration_of_Symptoms_Hours"],
    bins=[0, 6, 12, 24, 48, 1000],
    labels=["Very_Recent", "Recent", "12-24h", "Moderate", "Prolonged"],
    include_lowest=True
)


df_feat["Young_patient"] = (df_feat["Age"] <= 18).astype(int)
df_feat["High_CRP"] = (df_feat["CRP_Level_mg_L"] >= df_feat["CRP_Level_mg_L"].median()).astype(int)
df_feat["High_WBC"] = (df_feat["WBC_Count_k_uL"] >= df_feat["WBC_Count_k_uL"].median()).astype(int)

KeyError: "['Psoas_sign'] not in index"